In [1]:
import glob, joblib, numpy as np, pandas as pd, torch, torch.nn as nn
NB = glob.glob('/kaggle/input/notebooks/aadiparkhi/03a*')[0]
OUT = glob.glob('/kaggle/input/notebooks/aadiparkhi/04*')[0]
BK, K = ['dino', 'convnext'], 32
rng = np.random.default_rng(0)
D = {}
for d in ['places', 'bdd']:
    z = np.load(f'{NB}/feats_{d}.npz'); m = pd.read_csv(f'{NB}/meta_{d}.csv')
    D[d] = {'meta': m, **{k: {'emb': z[f'{k}_emb'].astype(np.float32), 'log': z[f'{k}_log']} for k in BK}}

OBJ = joblib.load(f'{OUT}/detector_objects.joblib'); M = joblib.load(f'{OUT}/phase4_models.joblib')

def make_ae(dim):
    return nn.Sequential(nn.Linear(dim, 256), nn.ReLU(), nn.Linear(256, 32), nn.ReLU(),
                         nn.Linear(32, 256), nn.ReLU(), nn.Linear(256, dim))

PP = {}
for (dd, k), o in OBJ.items():
    z = (D[dd][k]['emb'] - o['mu']) / o['sd']
    ae = make_ae(o['dim']); ae.load_state_dict(o['ae']); ae.eval()
    with torch.no_grad():
        x = torch.tensor(z, dtype=torch.float32); err = ((ae(x) - x) ** 2).mean(1).numpy()
    PP[(dd, k)] = (o['pca'].transform(z), o['S'], err)
    print(dd, k, 'val AE error:', round(err[(D[dd]['meta'].set == 'val').values].mean(), 4))

places dino val AE error: 0.4048
places convnext val AE error: 0.4516
bdd dino val AE error: 0.3838
bdd convnext val AE error: 0.4494


In [2]:
def ks_stat(X, S):
    B, n = len(X), len(S); Xs = np.sort(X, 0)
    Fr = np.stack([np.searchsorted(S[:, j], Xs[:, j], side='right') for j in range(K)], 1) / n
    up, lo = np.arange(1, B + 1)[:, None] / B, np.arange(B)[:, None] / B
    return np.maximum(np.abs(up - Fr), np.abs(lo - Fr)).max(0).mean()

def batches(rows, B, nb): return [rng.choice(rows, B) for _ in range(nb)]

def sc(d, b):
    out = []
    for k in BK:
        P, S, err = PP[(d, k)]; out += [ks_stat(P[b], S), err[b].mean()]
    return np.array(out)

CFG = {'dino': [0, 1], 'convnext': [2, 3], 'both': [0, 1, 2, 3]}

def calib_full(dd, n=2000):
    t = np.where((D[dd]['meta'].set == 'test').values)[0]
    A = np.array([sc(dd, b) for b in batches(t, 100, n)])
    return {k: np.quantile(A[:, c], 1 - 0.01 / len(c), axis=0) for k, c in CFG.items()}

def far_check(dd, n=2000):
    t = np.where((D[dd]['meta'].set == 'test').values)[0]; rng.shuffle(t); h = len(t) // 2
    S = lambda pool, n: np.array([sc(dd, b) for b in batches(pool, 100, n)])
    A, A2, B = S(t[:h], n), S(t[:h], 400), S(t[h:], 400)
    for k, c in CFG.items():
        th = np.quantile(A[:, c], 1 - 0.01 / len(c), axis=0)
        f = lambda X: round((X[:, c] > th).any(1).mean(), 3)
        print(dd, k, "same half:", f(A2), "| other half:", f(B))

THR = {dd: calib_full(dd) for dd in ('places', 'bdd')}
print({dd: {k: v.round(3).tolist() for k, v in THR[dd].items()} for dd in THR})
for dd in ('places', 'bdd'): far_check(dd)

ART = dict(thr=THR, reg=M['reg'], att=M['att'], C=M['C'], acc_c=M['acc_c'], pref=M['pref'])
joblib.dump(ART, '/kaggle/working/drift_artifacts.joblib')
print("saved:", list(ART))

{'places': {'dino': [0.102, 0.516], 'convnext': [0.102, 0.579], 'both': [0.104, 0.521, 0.102, 0.587]}, 'bdd': {'dino': [0.106, 0.485], 'convnext': [0.107, 0.551], 'both': [0.107, 0.487, 0.108, 0.554]}}
places dino same half: 0.025 | other half: 0.01
places convnext same half: 0.008 | other half: 0.012
places both same half: 0.022 | other half: 0.012
bdd dino same half: 0.01 | other half: 0.005
bdd convnext same half: 0.012 | other half: 0.042
bdd both same half: 0.012 | other half: 0.038
saved: ['thr', 'reg', 'att', 'C', 'acc_c', 'pref']


In [3]:
from scipy.optimize import nnls
ACT = 0.05     # predicted accuracy loss >= this -> "act"

NAMES = ['clear_day', 'clear_night', 'clear_twilight', 'overcast_day', 'overcast_twilight', 'rainy_day', 'rainy_night',
         'rainy_twilight', 'snow_fog_day', 'snow_fog_night', 'snow_fog_twilight']
GROUPS = {'night': lambda n: n.endswith('night'), 'day': lambda n: n.endswith('_day'), 'twilight': lambda n: n.endswith('twilight'),
          'rain': lambda n: n.startswith('rainy'), 'snow_fog': lambda n: n.startswith('snow_fog'), 'overcast': lambda n: n.startswith('overcast')}
m = D['bdd']['meta']; cond = m.cond_id.values
pred = D['bdd']['dino']['log'][:, :11].argmax(1)
pref, C, acc_c = ART['pref'], ART['C'], ART['acc_c']
tst = np.where((m.set == 'test').values)[0]

def draw(pool, p, B=100):
    cl = rng.choice(11, B, p=p); idx = np.empty(B, int)
    for c in range(11):
        w = np.where(cl == c)[0]
        if len(w): idx[w] = rng.choice(pool[cond[pool] == c], len(w))
    return idx

def est_change(b):
    q = np.bincount(pred[b], minlength=11) / len(b)
    w, _ = nnls(C, q); w /= w.sum()
    return w @ acc_c - pref @ acc_c

def ks_vec(X, S):
    B, n = len(X), len(S); Xs = np.sort(X, 0)
    Fr = np.stack([np.searchsorted(S[:, j], Xs[:, j], side='right') for j in range(K)], 1) / n
    up, lo = np.arange(1, B + 1)[:, None] / B, np.arange(B)[:, None] / B
    return np.maximum(np.abs(up - Fr), np.abs(lo - Fr)).max(0)

def fe(dd, b):
    o = []
    for k in BK:
        P, S, err = PP[(dd, k)]; x = P[b]; kv = ks_vec(x, S)
        o.append(np.concatenate([x.mean(0), x.std(0), kv, [kv.mean(), err[b].mean()]]))
    return np.concatenate(o)

def tier(dd, b):
    f = sc(dd, b)[CFG['both']] > ART['thr'][dd]['both']
    by = [k for k, hit in (('dino', f[:2].any()), ('convnext', f[2:].any())) if hit]
    if not by: return dict(level='none', fired_by=[], cause=None, impact=0.0)
    x = fe(dd, b)[None]; cause = ART['att'][dd].predict(x)[0]
    impact = est_change(b) if cause == 'mix_shift' else -ART['reg'][dd]['both'].predict(x)[0]
    return dict(level='act' if impact <= -ACT else 'watch', fired_by=by, cause=cause, impact=round(float(impact), 3))

rows = []
def go(dd, name, mk, n=20):
    r = [tier(dd, mk()) for _ in range(n)]
    L = pd.Series([x['level'] for x in r]).value_counts(normalize=True)
    rows.append(dict(domain=dd, scenario=name, none=L.get('none', 0), watch=L.get('watch', 0), act=L.get('act', 0),
                     impact=np.mean([x['impact'] for x in r if x['level'] != 'none'] or [0]),
                     cause=pd.Series([x['cause'] for x in r if x['cause']]).mode().get(0, '-')))

go('bdd', 'reference', lambda: draw(tst, pref))
for g, f_ in GROUPS.items():
    u = np.array([float(f_(nm)) for nm in NAMES]); u /= u.sum()
    go('bdd', f'{g} mix 0.75', lambda u=u: draw(tst, 0.25 * pref + 0.75 * u))
pt = np.where((D['places']['meta'].set == 'test').values)[0]
go('places', 'reference', lambda: rng.choice(pt, 100))
for dd, cs in (('bdd', ('brightness', 'fog', 'defocus_blur')), ('places', ('brightness', 'jpeg_compression', 'fog', 'defocus_blur'))):
    mm = D[dd]['meta']
    for c in cs:
        for s in (1, 3, 5):
            idx = np.where(((mm.corruption == c) & (mm.severity == s)).values)[0]
            go(dd, f'{c} s{s}', lambda idx=idx: rng.choice(idx, 100))
print(pd.DataFrame(rows).round(2).to_string(index=False))

domain            scenario  none  watch  act  impact            cause
   bdd           reference  1.00   0.00 0.00    0.00                -
   bdd      night mix 0.75  0.00   1.00 0.00    0.05        mix_shift
   bdd        day mix 0.75  0.25   0.75 0.00    0.02        mix_shift
   bdd   twilight mix 0.75  0.05   0.00 0.95   -0.16        mix_shift
   bdd       rain mix 0.75  0.00   0.45 0.55   -0.05        mix_shift
   bdd   snow_fog mix 0.75  0.00   1.00 0.00    0.01        mix_shift
   bdd   overcast mix 0.75  0.00   0.00 1.00   -0.12        mix_shift
places           reference  1.00   0.00 0.00    0.00                -
   bdd       brightness s1  0.05   0.85 0.10   -0.02       brightness
   bdd       brightness s3  0.00   0.00 1.00   -0.09       brightness
   bdd       brightness s5  0.00   0.00 1.00   -0.22       brightness
   bdd              fog s1  0.00   0.00 1.00   -0.09              fog
   bdd              fog s3  0.00   0.00 1.00   -0.22              fog
   bdd              

In [4]:
import json, sys, importlib, sklearn, scipy

CODE = '''# Drift monitor: DINOv2 (deployed model) + ConvNeXt (canary sensor). Batch-level alert from raw embeddings.
import numpy as np, joblib, torch, torch.nn as nn
from scipy.optimize import nnls

BK, ACT, REF_B = ['dino', 'convnext'], 0.05, 100

def _ks_vec(X, S):
    B, n = len(X), len(S); Xs = np.sort(X, 0)
    Fr = np.stack([np.searchsorted(S[:, j], Xs[:, j], side='right') for j in range(S.shape[1])], 1) / n
    up, lo = np.arange(1, B + 1)[:, None] / B, np.arange(B)[:, None] / B
    return np.maximum(np.abs(up - Fr), np.abs(lo - Fr)).max(0)

def _ae(dim):
    return nn.Sequential(nn.Linear(dim, 256), nn.ReLU(), nn.Linear(256, 32), nn.ReLU(),
                         nn.Linear(32, 256), nn.ReLU(), nn.Linear(256, dim))

class DriftMonitor:
    def __init__(self, objects_path, artifacts_path):
        self.obj, self.art = joblib.load(objects_path), joblib.load(artifacts_path)
        self.ae = {}
        for key, o in self.obj.items():
            ae = _ae(o['dim']); ae.load_state_dict(o['ae']); self.ae[key] = ae.eval()

    def _sensor(self, dd, k, emb):
        o = self.obj[(dd, k)]
        z = (np.asarray(emb, np.float32) - o['mu']) / o['sd']
        P = o['pca'].transform(z)
        with torch.no_grad():
            x = torch.tensor(z, dtype=torch.float32); err = ((self.ae[(dd, k)](x) - x) ** 2).mean(1).numpy()
        kv = _ks_vec(P, o['S'])
        return kv.mean(), err.mean(), np.concatenate([P.mean(0), P.std(0), kv, [kv.mean(), err.mean()]])

    def _label_shift(self, cond_logits):
        pred = np.asarray(cond_logits)[:, :11].argmax(1)
        q = np.bincount(pred, minlength=11) / len(pred)
        w, _ = nnls(self.art['C'], q); w /= w.sum()
        return w @ self.art['acc_c'] - self.art['pref'] @ self.art['acc_c']

    def assess(self, domain, emb, cond_logits=None):
        # emb: {'dino': (B, 768), 'convnext': (B, 768)} raw embeddings of one batch (same extraction as notebook 03a).
        # cond_logits: DINOv2 condition-head logits (B, >=11), BDD only; used for the label-shift impact estimate.
        parts = [self._sensor(domain, k, emb[k]) for k in BK]
        s = np.array([parts[0][0], parts[0][1], parts[1][0], parts[1][1]])
        fired = s > self.art['thr'][domain]['both']
        by = [k for k, hit in (('dino', fired[:2].any()), ('convnext', fired[2:].any())) if hit]
        B = len(emb[BK[0]])
        out = dict(level='none', fired_by=by, cause=None, impact=0.0, scores=s.round(4).tolist(), batch_size=B)
        if B != REF_B:
            out['note'] = f'thresholds were calibrated for batches of {REF_B} images; got {B}, so treat the result as indicative'
        if not by: return out
        x = np.concatenate([p[2] for p in parts])[None]
        cause = self.art['att'][domain].predict(x)[0]
        if cause == 'mix_shift' and cond_logits is not None: impact = self._label_shift(cond_logits)
        else: impact = -self.art['reg'][domain]['both'].predict(x)[0]
        out.update(level='act' if impact <= -ACT else 'watch', cause=cause, impact=round(float(impact), 3))
        return out
'''
open('/kaggle/working/drift_pipeline.py', 'w').write(CODE)
json.dump(dict(python=sys.version.split()[0], numpy=np.__version__, pandas=pd.__version__, scipy=scipy.__version__,
               scikit_learn=sklearn.__version__, torch=torch.__version__), open('/kaggle/working/versions.json', 'w'))

sys.path.insert(0, '/kaggle/working'); import drift_pipeline; importlib.reload(drift_pipeline)
mon = drift_pipeline.DriftMonitor(f'{OUT}/detector_objects.joblib', '/kaggle/working/drift_artifacts.joblib')

tests = [('bdd', lambda: draw(tst, pref))] * 10 + [('places', lambda: rng.choice(pt, 100))] * 10
for g in ('night', 'twilight', 'overcast'):
    u = np.array([float(GROUPS[g](nm)) for nm in NAMES]); u /= u.sum()
    tests += [('bdd', lambda u=u: draw(tst, 0.25 * pref + 0.75 * u))] * 10
for dd, c, s in (('bdd', 'fog', 3), ('bdd', 'brightness', 1), ('places', 'fog', 1), ('places', 'jpeg_compression', 3)):
    mm = D[dd]['meta']; idx = np.where(((mm.corruption == c) & (mm.severity == s)).values)[0]
    tests += [(dd, lambda idx=idx: rng.choice(idx, 100))] * 10

bad = 0
for dd, mk in tests:
    b = mk(); a = tier(dd, b)
    r = mon.assess(dd, {k: D[dd][k]['emb'][b] for k in BK}, D['bdd']['dino']['log'][b] if dd == 'bdd' else None)
    bad += not ((a['level'], a['cause']) == (r['level'], r['cause']) and abs(a['impact'] - r['impact']) < 2e-3)
print(f"{len(tests)} batches compared, mismatches: {bad}")
bb = draw(tst, pref)[:30]
print(mon.assess('bdd', {k: D['bdd'][k]['emb'][bb] for k in BK}))          # small-batch note
import os; print(sorted(os.listdir('/kaggle/working')))

90 batches compared, mismatches: 0
{'level': 'watch', 'fired_by': ['dino', 'convnext'], 'cause': 'mix_shift', 'impact': -0.018, 'scores': [0.1413, 0.4398, 0.1521, 0.5372], 'batch_size': 30, 'note': 'thresholds were calibrated for batches of 100 images; got 30, so treat the result as indicative'}
['__notebook__.ipynb', '__pycache__', 'drift_artifacts.joblib', 'drift_pipeline.py', 'versions.json']


In [5]:
def calib_B(dd, B, n=1000):
    t = np.where((D[dd]['meta'].set == 'test').values)[0]
    A = np.array([sc(dd, b) for b in batches(t, B, n)])
    return {k: np.quantile(A[:, c], 1 - 0.01 / len(c), axis=0) for k, c in CFG.items()}

def far_B(dd, B, n=1000):
    t = np.where((D[dd]['meta'].set == 'test').values)[0]; rng.shuffle(t); h = len(t) // 2
    S = lambda pool, n: np.array([sc(dd, b) for b in batches(pool, B, n)])
    A, E = S(t[:h], n), S(t[h:], 400); c = CFG['both']
    th = np.quantile(A[:, c], 1 - 0.01 / len(c), axis=0)
    print(dd, 'B =', B, '| false alarms with B=100 thresholds:', round((E[:, c] > THR[dd]['both']).any(1).mean(), 3),
          '| with B-specific thresholds:', round((E[:, c] > th).any(1).mean(), 3))

for dd in ('places', 'bdd'):
    for B in (30, 50): far_B(dd, B)
ART['thr_by_B'] = {dd: {B: calib_B(dd, B) for B in (30, 50, 100)} for dd in ('places', 'bdd')}
joblib.dump(ART, '/kaggle/working/drift_artifacts.joblib')
print({dd: {B: ART['thr_by_B'][dd][B]['both'].round(3).tolist() for B in (30, 50, 100)} for dd in ART['thr_by_B']})

places B = 30 | false alarms with B=100 thresholds: 1.0 | with B-specific thresholds: 0.015
places B = 50 | false alarms with B=100 thresholds: 1.0 | with B-specific thresholds: 0.012
bdd B = 30 | false alarms with B=100 thresholds: 1.0 | with B-specific thresholds: 0.005
bdd B = 50 | false alarms with B=100 thresholds: 1.0 | with B-specific thresholds: 0.03
{'places': {30: [0.182, 0.551, 0.185, 0.623], 50: [0.142, 0.534, 0.143, 0.602], 100: [0.102, 0.516, 0.101, 0.583]}, 'bdd': {30: [0.181, 0.524, 0.186, 0.581], 50: [0.147, 0.501, 0.144, 0.566], 100: [0.107, 0.485, 0.108, 0.551]}}


In [6]:
src = open('/kaggle/working/drift_pipeline.py').read()
old1 = """        fired = s > self.art['thr'][domain]['both']
"""
new1 = """        B = len(emb[BK[0]]); tb = self.art.get('thr_by_B', {}).get(domain)
        if tb:
            sizes = sorted(tb); Bt = max([z for z in sizes if z <= B] or [sizes[0]]); thr = tb[Bt]['both']   # largest calibrated size <= B (conservative)
        else:
            Bt, thr = REF_B, self.art['thr'][domain]['both']
        fired = s > thr
"""
old2 = """        B = len(emb[BK[0]])
        out = dict(level='none', fired_by=by, cause=None, impact=0.0, scores=s.round(4).tolist(), batch_size=B)
        if B != REF_B:
            out['note'] = f'thresholds were calibrated for batches of {REF_B} images; got {B}, so treat the result as indicative'
"""
new2 = """        out = dict(level='none', fired_by=by, cause=None, impact=0.0, scores=s.round(4).tolist(), batch_size=B, thr_batch_size=Bt)
        notes = []
        if tb and B < min(tb): notes.append(f'batch is smaller than the smallest calibrated size ({min(tb)}); false alarms may exceed 1%')
        if B != REF_B: notes.append(f'impact models were trained on {REF_B}-image batches; treat the estimate as indicative')
        if notes: out['note'] = '; '.join(notes)
"""
assert old1 in src and old2 in src
open('/kaggle/working/drift_pipeline.py', 'w').write(src.replace(old1, new1).replace(old2, new2))

for dd in ART['thr_by_B']: ART['thr_by_B'][dd][100] = THR[dd]      # B=100 keeps the 2000-batch thresholds used by tier()
joblib.dump(ART, '/kaggle/working/drift_artifacts.joblib')
importlib.reload(drift_pipeline)
mon = drift_pipeline.DriftMonitor(f'{OUT}/detector_objects.joblib', '/kaggle/working/drift_artifacts.joblib')

bad = 0
for dd, mk in tests:
    b = mk(); a = tier(dd, b)
    r = mon.assess(dd, {k: D[dd][k]['emb'][b] for k in BK}, D['bdd']['dino']['log'][b] if dd == 'bdd' else None)
    bad += not ((a['level'], a['cause']) == (r['level'], r['cause']) and abs(a['impact'] - r['impact']) < 2e-3)
print(f"B=100 equivalence: {len(tests)} batches, mismatches: {bad}")

for dd in ('places', 'bdd'):
    pool = np.where((D[dd]['meta'].set == 'test').values)[0]
    for B in (30, 50, 100):
        fa = np.mean([mon.assess(dd, {k: D[dd][k]['emb'][b] for k in BK})['level'] != 'none' for b in batches(pool, B, 200)])
        print(dd, 'B =', B, '| false alarms through the module:', round(fa, 3))

B=100 equivalence: 90 batches, mismatches: 0
places B = 30 | false alarms through the module: 0.005
places B = 50 | false alarms through the module: 0.015
places B = 100 | false alarms through the module: 0.005
bdd B = 30 | false alarms through the module: 0.005
bdd B = 50 | false alarms through the module: 0.02
bdd B = 100 | false alarms through the module: 0.015
